# Sample and null permute portion of every conversation.

Requested by @RickDale

In [1]:
import os
import numpy as np
import pandas as pd
import re
import shutil

import pyarrow.parquet as pq
from tqdm.notebook import tqdm
import warnings;warnings.filterwarnings('ignore')

In [2]:
SERVER_READY_DATA = '../data/server_ready'

OUTPUT_FOLDER = '../data/null-perm'
if not os.path.exists(OUTPUT_FOLDER):
    os.mkdir(OUTPUT_FOLDER)

In [3]:
def grab_all_files(PATH, file_type='.parquet'):
    files = [
        [
            os.path.join(root, f) for f in files
            if f.endswith(file_type) and (not f.startswith('.'))
        ]
        for root, _, files in os.walk(PATH)
    ]
    return sum(files, [])

## Iterate, permute, sample

In [4]:
fs = grab_all_files(SERVER_READY_DATA)
len(fs)

26

In [5]:
sample_frac = .05

In [7]:
families = set([f.split('/')[-1].split('-')[1] for f in fs])
families

{'callfriend', 'callhome', 'croatian', 'finchat_corpus.parquet', 'yiddish'}

### Data Null

In [11]:
for corpus in tqdm(families):
    fs_ = [f for f in fs if (corpus in f)]
    output_ = os.path.join(OUTPUT_FOLDER, '{}-null_perm.parquet').format(corpus)

    df = pd.concat(
        [pd.read_parquet(f) for f in fs_],
        ignore_index=True
    )

    if 'yiddish' in df['corpus'].unique():
        df['corpus'] = 'yiddish-yid'

    df['lang'] = [corpus.split('-')[1] for corpus in df['corpus']]

    df['y_utterance'] = df.groupby(by=['lang'])['y_utterance'].transform(np.random.permutation)

    output_data = []
    groups = df.groupby(by=['corpus', 'conversation_id', 'x_speaker', 'x_turn_id'])
    for labels, dfi in groups:
        output_data += [dfi.sample(n=int(np.ceil(sample_frac * len(dfi))))]

    pd.concat(output_data).to_parquet(output_, engine='fastparquet', compression='snappy')

  0%|          | 0/5 [00:00<?, ?it/s]

## Sort and send

In [12]:
DATA = [f for f in os.listdir(OUTPUT_FOLDER) if (not f.startswith('.'))]

In [13]:
to_rosen, to_itkin = [], []
to_rosen += np.random.choice(DATA, size=(int(np.ceil(len(DATA)/2)),), replace=False).tolist()
to_itkin = [f for f in DATA if f not in to_rosen]
len(to_itkin), len(to_rosen)

(2, 3)

In [14]:
ROSEN_PATH = os.path.join(OUTPUT_FOLDER, 'to_rosen')
if not os.path.exists(ROSEN_PATH):
    os.mkdir(ROSEN_PATH)

for f in tqdm(to_rosen):
    shutil.move(
        os.path.join(OUTPUT_FOLDER, f),
        os.path.join(ROSEN_PATH, f)
    )

  0%|          | 0/3 [00:00<?, ?it/s]

In [15]:
ITKIN_PATH = os.path.join(OUTPUT_FOLDER, 'to_itkin')
if not os.path.exists(ITKIN_PATH):
    os.mkdir(ITKIN_PATH)

for f in tqdm(to_itkin):
    shutil.move(
        os.path.join(OUTPUT_FOLDER, f),
        os.path.join(ITKIN_PATH, f)
    )

  0%|          | 0/2 [00:00<?, ?it/s]